In [ ]:

import pandas as pd
import numpy as np
import ast

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


num_cols = ['budget', 'popularity', 'runtime', 'vote_average', 'vote_count']

for col in num_cols:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)


median_budget = train.loc[train['budget'] > 0, 'budget'].median()
train['budget'] = train['budget'].replace(0, median_budget)
test['budget'] = test['budget'].replace(0, median_budget)


def count_items(x):
    try:
        return len(ast.literal_eval(x))
    except:
        return 0

for df in [train, test]:
    df['genre_count'] = df['genres'].apply(count_items)
    df['keyword_count'] = df['keywords'].apply(count_items)
    df['company_count'] = df['production_companies'].apply(count_items)
    df['country_count'] = df['production_countries'].apply(count_items)
    df['language_count'] = df['spoken_languages'].apply(count_items)

    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['release_year'] = df['release_date'].dt.year
    df['release_month'] = df['release_date'].dt.month

    
    df['log_budget'] = np.log1p(df['budget'])
    df['budget_runtime_ratio'] = df['budget'] / (df['runtime'] + 1)

lang_freq = train['original_language'].value_counts().to_dict()
train['lang_freq'] = train['original_language'].map(lang_freq)
test['lang_freq'] = test['original_language'].map(lang_freq).fillna(1)

y = np.log1p(train['revenue'])


features = [
    'budget', 'log_budget', 'budget_runtime_ratio',
    'popularity', 'runtime',
    'vote_average', 'vote_count',
    'genre_count', 'keyword_count',
    'company_count', 'country_count', 'language_count',
    'release_year', 'release_month',
    'lang_freq'
]

X = train[features]
X_test = test[features]


kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_preds = np.zeros(len(test))
rf_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Training fold {fold+1}/5")

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]

    
    xgb = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.015,
        max_depth=5,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.15,
        reg_alpha=0.15,
        reg_lambda=1.2,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )

    
    rf = RandomForestRegressor(
        n_estimators=600,
        max_depth=18,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_tr, y_tr)
    rf.fit(X_tr, y_tr)

    xgb_preds += xgb.predict(X_test) / kf.n_splits
    rf_preds += rf.predict(X_test) / kf.n_splits


test_log_preds = 0.7 * xgb_preds + 0.3 * rf_preds
test_preds = np.expm1(test_log_preds)

\
test_preds = np.maximum(test_preds, 0)


cap = np.percentile(train['revenue'], 99.3)
test_preds = np.minimum(test_preds, cap)


submission = pd.DataFrame({
    'id': test['id'],
    'revenue': test_preds
})

submission.to_csv("submission_final.csv", index=False)
print("✅ submission_final.csv saved successfully")
